# Process DB in 'raw' directory to make necessary columns
- SMILES_PubChem
- can_SMILES

In [ ]:
import pandas as pd
import time
from tqdm import tqdm
import numpy as np
import os
import logging
import time
import matplotlib.pyplot as plt
import re
from rdkit import Chem
from thermo.chemical import Chemical
import pubchempy as pcp

## Skin Permeability Exp.

In [ ]:
# Read all sheets from Excel file into a dictionary of dataframes
filename = 'SkinPerm_exp_250707'
excel_file = pd.ExcelFile(f'../data/raw/{filename}.xlsx')
sheet_names = excel_file.sheet_names

In [ ]:
sheet_names

In [ ]:
# Use first sheet as default dataframe
df = pd.read_excel(excel_file, sheet_name=sheet_names[0])
df

In [ ]:
len(df[df['SMILES_PubChem']=='Not found in PubChem'])

In [ ]:
def safe_canonical_smiles(x):
    if x == 'Not found in PubChem' or pd.isna(x):
        return None
    mol = Chem.MolFromSmiles(x)
    if mol is None:
        print(f"Invalid SMILES: {x}")
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True)

In [ ]:
# Canonicalize SMILES_PubChem using RDKit (ignoring chiral information)
df['can_SMILES'] = df['SMILES_PubChem'].apply(lambda x: safe_canonical_smiles(x) if x != 'Not found in PubChem' else None)
df

In [ ]:
df[df.can_SMILES.isna()]

In [ ]:
len(set(df.Compound_Name))

In [ ]:
len(set(df.SMILES_PubChem))

In [ ]:
len(set(df.can_SMILES))

In [ ]:
df

In [ ]:
df.to_csv(f'../data/processed/{filename}_processed.csv', index=False)

## Self Diffusion Coefficient Exp.

In [ ]:
# Read all sheets from Excel file into a dictionary of dataframes
excel_file = pd.ExcelFile(r'../data/raw/SelfDiff_exp_iecr_2c03342_si_001.xlsx')
sheet_names = excel_file.sheet_names

In [ ]:
sheet_names

In [ ]:
# Use first sheet as default dataframe
df = pd.read_excel(excel_file, sheet_name='Table S2')
df

In [ ]:
df = df.rename(columns={
    'Substance': 'Compound_Name',
    'T/K': 'T_K'
})
df

In [ ]:
# Get SMILES from compound names using PubChemPy
def get_smiles_from_name(name):
    try:
        compounds = pcp.get_compounds(name, 'name')
        if compounds:
            return compounds[0].isomeric_smiles
        return None
    except Exception as e:
        print(f"Error retrieving SMILES for {name}: {e}")
        return None

In [ ]:
smiles_pubchem = []
for name in tqdm(df['Compound_Name'], desc='Getting SMILES from PubChem'):
    smiles_pubchem.append(get_smiles_from_name(name))
    time.sleep(0.2)  # To avoid hitting API rate limits

df['SMILES_PubChem'] = smiles_pubchem
df[['Compound_Name', 'SMILES_PubChem']].head()

In [ ]:
# get ref column values
df['ref'] = 'acs.iecr.2c03342'
df

In [ ]:
def safe_canonical_smiles(x):
    if x == 'Not found in PubChem' or pd.isna(x):
        return None
    mol = Chem.MolFromSmiles(x)
    if mol is None:
        print(f"Invalid SMILES: {x}")
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True)

In [ ]:
# Canonicalize SMILES_PubChem using RDKit (ignoring chiral information)
df['can_SMILES'] = df['SMILES_PubChem'].apply(lambda x: safe_canonical_smiles(x) if x != 'Not found in PubChem' else None)
df

In [ ]:
df[df.can_SMILES.isna()]

In [ ]:
len(set(df.can_SMILES))

In [ ]:
# current_date = time.strftime("%Y%m%d")
df.to_csv(f'../data/processed/SelfDiff_exp_processed.csv', index=False)

# ChEMBL v34 (batch 1)

In [ ]:
import random
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors

# ---------- helper criteria ----------
ALLOWED_ELEMENTS = {"H", "C", "N", "O", "F", "S", "Cl", "Br", "I", "P"}

def passes_filters(mol):
    """FF-friendly window."""
    if any(a.GetSymbol() not in ALLOWED_ELEMENTS for a in mol.GetAtoms()):
        return False
    if Chem.GetFormalCharge(mol) != 0:
        return False
    mw = Descriptors.MolWt(mol)
    if not (100 <= mw <= 600):
        return False
    logp = Crippen.MolLogP(mol)
    if not (-2 <= logp <= 6):
        return False
    return True

def bin_key(mol):
    """Coarse (MW, logP, rotatable-bond) bin for even sampling."""
    return (
        int(Descriptors.MolWt(mol) // 50),                        # MW 50-Da buckets
        int((Crippen.MolLogP(mol) + 10) // 1),                   # logP 1-unit buckets
        int(rdMolDescriptors.CalcNumRotatableBonds(mol) // 2),   # rotB 2-bond buckets
    )

# ---------- main function ----------
def filter_and_sample(df, sample_size=1000, seed=2025, write_smi=None):
    """
    df              : pandas DataFrame with 'SMILES_chembl' and 'chembl_id' columns
    sample_size     : number of molecules to return
    write_smi       : optional path -> write plain .smi file
    returns         : filtered & sampled DataFrame
    """
    random.seed(seed)
    unique = {}  # canonical_smiles -> (chembl_id, Mol)

    for smi, cid in tqdm(zip(df["SMILES_chembl"], df["chembl_id"]),
                             total=len(df), desc="Screening"):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        # keep largest fragment and sanitise
        mol = max(Chem.GetMolFrags(mol, asMols=True), key=lambda m: m.GetNumAtoms())
        try:
            Chem.SanitizeMol(mol)
        except ValueError:
            continue
        if not passes_filters(mol):
            continue
        can = Chem.MolToSmiles(mol, isomericSmiles=True)
        if can not in unique:               # de-duplicate across dataframe
            unique[can] = (cid, mol)

    # diversity sampling
    bins = {}
    for smi_can, (cid, mol) in unique.items():
        bins.setdefault(bin_key(mol), []).append((smi_can, cid))

    per_bin = max(1, sample_size // len(bins))
    selected = []
    for lst in bins.values():
        random.shuffle(lst)
        selected.extend(lst[:per_bin])

    if len(selected) > sample_size:         # overshoot trim
        selected = random.sample(selected, sample_size)

    # assemble output dataframe
    out_df = pd.DataFrame(
        {
            "smiles": [s for s, _ in selected],
            "chembl_id": [c for _, c in selected],
        }
    )

    if write_smi:
        out_df.to_csv(write_smi, sep=" ", header=False, index=False)

    return out_df

In [ ]:
df_raw = pd.read_csv("../data/raw/chembl_34_chemreps.txt",
                     sep="\t", usecols=["canonical_smiles", "chembl_id"]
).rename(columns={"canonical_smiles": "SMILES_chembl"})

sampled = filter_and_sample(df_raw,
                            sample_size=1000,
                            write_smi="chembl34_batch1.smi")  # optional file

print(sampled.head())

In [ ]:
sampled

In [ ]:
# get ref column values
sampled['ref'] = 'chembl34_batch1'
sampled

In [ ]:
def safe_canonical_smiles(x):
    if x == 'Not found in PubChem' or pd.isna(x):
        return None
    mol = Chem.MolFromSmiles(x)
    if mol is None:
        print(f"Invalid SMILES: {x}")
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True)

In [ ]:
# Canonicalize SMILES_PubChem using RDKit (ignoring chiral information)
sampled['can_SMILES'] = sampled['smiles'].apply(lambda x: safe_canonical_smiles(x) if x != 'Not found in PubChem' else None)
sampled

In [ ]:
sampled.to_csv(f'../data/processed/chembl_batch1_processed.csv', index=False)